# 00 · Environment & data check

A 30-second smoke test: run it top to bottom and, if every cell is green, your environment is ready for the rest of the tutorials. It checks the three things that actually break: the **install** (can you import `roman_disperser` and is JAX working?), the **reference data** (is it hydrated, and — crucially — can the *kernel* find it?), and a **tiny dispersion** end to end.

If something here fails, fix it before moving on; `docs/SETUP.md` is the canonical setup guide. This notebook teaches nothing about the science — it just proves the plumbing works.

## 1 · Install and JAX backend

We import the pieces the tutorials use and print versions. JAX should report a backend (`cpu` on a laptop, `gpu` on a GPU node) and at least one device; either is fine.

In [ ]:
import warnings
warnings.filterwarnings("ignore")          # keep this smoke test free of library noise

from importlib.metadata import version
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm
import jax
import jax.numpy as jnp

from roman_disperser import paths, refdata, psf_model, star_disperser
import roman_disperser.optical_model_jax as omj
from roman_disperser.optical_model import RomanOpticalModel
import tutorial_helpers as th

print("roman_disperser :", version("roman_disperser"))
print("jax             :", version("jax"))
print(f"\n✅ imports OK — JAX backend '{jax.default_backend()}', {len(jax.devices())} device(s)")

## 2 · Reference data

`roman_disperser` finds its reference data via `paths.data_dir()`. The check below prints what the **kernel** resolves and whether each asset is hydrated. If anything is `MISSING` (or the path looks wrong), see [`docs/SETUP.md`](../docs/SETUP.md) — usually the data just needs to be visible to the *kernel*, not only your shell.

In [ ]:
import os

print("ROMAN_DISPERSER_DATA :", os.environ.get("ROMAN_DISPERSER_DATA", "(unset)"))
print("PIXI_PROJECT_ROOT    :", os.environ.get("PIXI_PROJECT_ROOT", "(unset)"))
print("resolved data_dir    :", paths.data_dir(), "\n")

assets = {
    "optical model": paths.optical_model_path(),
    "sensitivities": paths.sensitivity_dir(),
    "synphot      ": paths.synphot_dir(),
    "psf_cache    ": paths.psf_cache_dir(),
    "catalogs     ": paths.catalog_dir(),
}
missing = []
for name, p in assets.items():
    p = Path(p)
    n = len(list(p.glob("*"))) if p.is_dir() else (1 if p.exists() else 0)
    print(f"  {name}: {'OK ' if n else 'MISSING'} ({n} item(s))")
    if not n:
        missing.append(name.strip())

if missing:
    print(f"\n❌ missing: {', '.join(missing)} — run `pixi run hydrate` or see docs/SETUP.md")
else:
    print("\n✅ all reference data hydrated")

## 3 · A smoke dispersion

Finally, disperse a single flat-spectrum star on SCA 5, order 1 — exercising the optical model, the PSF cache, and the JIT compile path. The first call compiles (a few seconds); we just need a nonzero image out.

In [ ]:
SCA = 5
model = RomanOpticalModel(config_file=str(paths.optical_model_path()))
opt = omj.make_sca_payload(model, sca=SCA, order="1")
psf = psf_model.get_or_make_psf_payload(detector=f"WFI{SCA:02d}", order="1",
                                        cache_dir=str(paths.psf_cache_dir()), verbose=False)
disperse = star_disperser.make_star_disperser(psf, opt)

wl = jnp.asarray(th.grism_wavelength_grid()[0])         # 2 Å science grid; flat spectrum
out = np.asarray(disperse(2000.0, 2000.0, wl, jnp.ones_like(wl),
                          jnp.zeros((4088, 4088), jnp.float32)))

assert out.sum() > 0, "dispersion produced an empty image — check the PSF cache / data"
ys, xs = np.nonzero(out)
cut = out[ys.min()-5:ys.max()+5, xs.min()-5:xs.max()+5]
fig, ax = plt.subplots(figsize=(3, 5))
ax.imshow(cut, origin="lower", cmap="inferno",
          norm=AsinhNorm(linear_width=cut.max()*0.01, vmin=0, vmax=cut.max()))
ax.set(title="smoke dispersion (SCA5, order 1)", xticks=[], yticks=[])
fig.tight_layout()
print(f"✅ smoke dispersion OK — total {out.sum():.0f} e⁻/s on the detector")

## All green?

Then you're set — start with **[01 · Spectra, bandpasses, and count rates](01_spectra_to_counts.ipynb)**. If the data check showed MISSING assets, run `pixi run hydrate` (developers) or follow `docs/SETUP.md` (users) and re-run this notebook.